In [1]:
import lxml.etree as ET
import json

In [2]:
path = "BankSystem_new_kdm.xmi"
tree = ET.parse(path)
root = tree.getroot()
ns = {
    'code': 'http://www.eclipse.org/MoDisco/kdm/code',
    'xsi': 'http://www.w3.org/2001/XMLSchema-instance'
}
PROJECT_FILTER = "com.bank.logic"

projet_data = {"projet": "BankSystem_KDM", "classes": []}

In [3]:
def resolve_kdm_path(path_str, xmi_root):
    """
    Correctly resolves KDM paths like /0/@model.0/@codeElement.1
    """
    if not path_str or not path_str.startswith('/'):
        return None
    
    parts = path_str.strip('/').split('/')
    # In MoDisco KDM, /0 refers to the first child of XMI (the kdm:Segment)
    try:
        current = xmi_root[0] 
    except IndexError:
        return None
    
    # Iterate through segments starting after '0'
    for part in parts[1:]:
        clean_part = part.replace('@', '')
        if '.' in clean_part:
            tag_name, index = clean_part.split('.')
            index = int(index)
        else:
            tag_name = clean_part
            index = 0
            
        # Filter children by local name (ignoring namespaces)
        children = [c for c in current if c.tag.split('}')[-1] == tag_name]
        
        if index < len(children):
            current = children[index]
        else:
            return None
    return current

In [4]:
def get_kdm_type(element, xmi_root):
    """
    Retrieves and cleans the type name (e.g., 'double' or 'List<String>').
    """
    path_val = element.get('type')
    target_node = resolve_kdm_path(path_val, xmi_root)
    
    if target_node is not None:
        name = target_node.get('name')
        if name:
            # Clean up the name (e.g., java.util.List -> List)
            clean_name = name.replace("java.lang.", "").replace("java.util.", "")
            return clean_name
    return "Object" # Fallback if resolution fails

In [5]:
def get_kdm_visibility(element, ns):
    """
    Extracts visibility from KDM export attributes.
    """
    export = element.xpath("./attribute[@tag='export']/@value", namespaces=ns)
    if export:
        val = export[0]
        return 'public' if val == 'public' else ('private' if val == 'private' else 'package')
    return 'package'

In [6]:
all_classes = root.xpath("//codeElement[@xsi:type='code:ClassUnit']", namespaces=ns)

for cls in all_classes:
    # Package Filter
    ancestors = cls.xpath("ancestor::codeElement[@xsi:type='code:Package']/@name", namespaces=ns)
    if PROJECT_FILTER not in ".".join(ancestors):
        continue

    class_info = {
        "name": cls.get('name'),
        "visibility": get_kdm_visibility(cls, ns),
        "attributs": [],
        "methods": []
    }

    # Extract Fields
    for f in cls.xpath("./codeElement[@xsi:type='code:StorableUnit']", namespaces=ns):
        class_info["attributs"].append({
            "name": f.get('name'),
            "type": get_kdm_type(f, root),
            "visibility": get_kdm_visibility(f, ns)
        })

    # Extract Methods
    for m in cls.xpath("./codeElement[@xsi:type='code:MethodUnit']", namespaces=ns):
        m_name = m.get('name')
        
        resolved_type = get_kdm_type(m, root)
        if resolved_type == m_name or resolved_type == "Object":
            m_type = "void"
        else:
            m_type = resolved_type
            
        params = m.xpath("./codeElement[@xsi:type='code:ParameterUnit']", namespaces=ns)
        if not params and m_name == "main":
            params_list = [{"name": "args", "type": "String[]"}]
        else:
            params_list = [{"name": p.get('name'), "type": get_kdm_type(p, root)} for p in params]

        class_info["methods"].append({
            "name": m_name,
            "type_retour": m_type,  # Use the validated m_type here
            "visibility": get_kdm_visibility(m, ns),
            "parametres": params_list,
            "is_constructor": (m_name == class_info["name"])
        })

    projet_data["classes"].append(class_info)

In [7]:
import os

def read_java_code(class_name, code_dir="code"):
    """
    Reads the Java source file corresponding to the class name.
    """
    file_path = os.path.join(code_dir, f"{class_name}.java")
    
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()
    else:
        return f"// Source code for {class_name} not found."


def prompt_generator(classe, code_dir="code"):
    class_name = classe['name']
    
    if class_name.endswith("Test"):
        return []
    
    source_code = read_java_code(class_name, code_dir)

    if "not found" in source_code:
        return []
    
    # Read source code
    source_code = read_java_code(class_name, code_dir)

    # 1. Context
    context = f"Context: You are an expert Java developer specializing in unit testing, edge-case analysis, and high coverage test design.\n"
    context += f"Your task is to generate a COMPLETE and HIGH-QUALITY JUnit 5 test class for '{class_name}' aiming for near 100% code coverage.\n\n"

    # 2. Class structure (KDM)
    context += "CLASS STRUCTURE (Fields):\n"
    if not classe['attributs']:
        context += "  - No fields defined.\n"
    else:
        for attr in classe['attributs']:
            context += f"  - {attr['visibility']} {attr['type']} {attr['name']}\n"

    # 3. Inject source code (CRITICAL)
    context += "\nORIGINAL SOURCE CODE:\n"
    context += "-------------------------------------\n"
    context += source_code + "\n"
    context += "-------------------------------------\n"

    # Anti-hallucination constraint
    context += "\nIMPORTANT:\n"
    context += "- You MUST base your tests strictly on the provided source code.\n"
    context += "- Do NOT assume behavior that is not explicitly defined.\n"
    context += "- Cover ALL branches, conditions, and edge cases.\n\n"

    prompts = []

    # 4. Methods
    for meth in classe['methods']:
        params_str = ", ".join([f"{p['type']} {p['name']}" for p in meth['parametres']])
        signature = f"{meth['type_retour']} {meth['name']}({params_str})"

        prompt = context

        prompt += "METHOD TO TEST:\n"
        prompt += f"  - Name: {meth['name']}\n"
        prompt += f"  - Visibility: {meth['visibility']}\n"
        prompt += f"  - Signature: {signature}\n"
        prompt += f"  - Constructor: {'Yes' if meth['is_constructor'] else 'No'}\n"
        prompt += f"  - Static: {'Yes' if meth.get('est_statique', False) else 'No'}\n\n"

        # 5. Special constraints
        prompt += "SPECIAL TESTING CONSTRAINTS (MANDATORY):\n"

        if meth['name'] == "main":
            prompt += "1. The method uses Scanner input.\n"
            prompt += "   → Simulate input using System.setIn(ByteArrayInputStream)\n"
            prompt += "   → Capture output using System.setOut(PrintStream)\n"
            prompt += "   → Reset System.in and System.out after each test\n"
        else:
            prompt += "1. Tests must be deterministic and isolated.\n"

        prompt += "2. Every test MUST validate real behavior (no trivial tests).\n"
        prompt += "3. Cover ALL branches (if/else, exceptions, edge cases).\n\n"

        # 6. Test design rules
        prompt += "TEST DESIGN REQUIREMENTS:\n"
        prompt += "1. Use JUnit 5:\n"
        prompt += "   - @Test\n"
        prompt += "   - @BeforeEach\n"
        prompt += "   - @AfterEach (if needed)\n"
        prompt += "   - @DisplayName (MANDATORY)\n\n"

        prompt += "2. @DisplayName MUST:\n"
        prompt += "   - Describe the scenario\n"
        prompt += "   - Include expected result\n"
        prompt += "   Example: \"Withdraw exceeding balance → throws InsolventException\"\n\n"

        prompt += "3. Follow AAA strictly:\n"
        prompt += "   - Arrange\n"
        prompt += "   - Act\n"
        prompt += "   - Assert\n\n"

        prompt += "4. Assertions:\n"
        prompt += "   - Use assertEquals, assertTrue, assertThrows, assertDoesNotThrow\n"
        prompt += "   - Validate ALL observable behavior (return values, state, output)\n\n"

        prompt += "5. Edge Cases (MANDATORY):\n"
        prompt += "   - Boundary values\n"
        prompt += "   - Invalid inputs\n"
        prompt += "   - Exception paths\n"
        prompt += "   - Special values (0, negative, large numbers)\n\n"

        prompt += "6. Coverage Goal:\n"
        prompt += "   - Cover ALL lines and branches\n"
        prompt += "   - Include multiple tests per scenario if needed\n\n"

        prompt += "7. Code Quality:\n"
        prompt += "   - Clean and readable\n"
        prompt += "   - No duplication\n"
        prompt += "   - Use helper methods if needed\n\n"

        prompt += "OUTPUT REQUIREMENTS:\n"
        prompt += "- ONLY Java code\n"
        prompt += "- No explanations\n"
        prompt += "- Fully compilable JUnit 5 class\n"

        prompts.append(prompt)

    return prompts

In [8]:
generatedPrompts = []
for cls in projet_data['classes']:
    liste_prompts = prompt_generator(cls)
    for pr in liste_prompts:
        generatedPrompts.append(pr)

In [9]:
from ollama import chat
from concurrent.futures import ThreadPoolExecutor
import time
import random

models = [
    "gemini-3-flash-preview",
    "deepseek-v3.2:cloud",
    "gemma4:31b-cloud"
]

AllTests = []

In [10]:
def clean_test_output(text):
    if not isinstance(text, str):
        return text

    lines = text.strip().splitlines()
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].startswith("```"):
        lines = lines[:-1]

    return "\n".join(lines).strip()

In [11]:
def run_prompt(model_name, prompt, index, total, max_retries=5):
    delay = 1

    for attempt in range(max_retries):
        try:
            print(f"[{model_name}] Prompt {index+1}/{total} (attempt {attempt+1})")

            response = chat(
                model=model_name,
                messages=[{'role': 'user', 'content': prompt}],
            )

            return clean_test_output(response.message.content), False  # success

        except Exception as e:
            error_msg = str(e)

            if "429" in error_msg or "timed out" in error_msg:
                if attempt < max_retries - 1:
                    sleep_time = delay + random.uniform(0, 1)
                    print(f"Retry {attempt+1}/{max_retries} in {sleep_time:.2f}s...")
                    time.sleep(sleep_time)
                    delay *= 2
                else:
                    print(f"Failed after {max_retries} retries")
                    return f"ERROR: Rate limit", True
            else:
                print(f"Non-retryable error: {error_msg}")
                return f"ERROR: {error_msg}", True

In [ ]:
for model_name in models:
    print(f"\nRunning model: {model_name}\n")

    total_prompts = len(generatedPrompts)
    results = [None] * total_prompts
    failed_tasks = []
    max_workers = 5

    while True:
        errors_count = 0

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = []

            for i, prompt in enumerate(generatedPrompts):
                if results[i] is None:  # only run missing ones
                    future = executor.submit(run_prompt, model_name, prompt, i, total_prompts)
                    futures.append((i, future))

            for i, future in futures:
                result, failed = future.result()

                if failed:
                    errors_count += 1
                    failed_tasks.append((i, generatedPrompts[i]))
                else:
                    results[i] = result

        error_ratio = errors_count / max(1, total_prompts)

        if error_ratio > 0.3 and max_workers > 2:
            max_workers -= 1
            print(f"High error rate ({error_ratio:.2f}) → Reducing workers to {max_workers}")
        else:
            break  
    if failed_tasks:
        print(f"\nRetrying {len(failed_tasks)} failed prompts sequentially...\n")

        for i, prompt in failed_tasks:
            result, _ = run_prompt(model_name, prompt, i, total_prompts)
            results[i] = result

    clean_name = model_name.split(":")[0]

    AllTests.append({
        "name": clean_name,
        "test": results
    })

print("\nAll models completed!")


Running model: gemini-3-flash-preview

[gemini-3-flash-preview] Prompt 1/16 (attempt 1)
[gemini-3-flash-preview] Prompt 2/16 (attempt 1)
[gemini-3-flash-preview] Prompt 3/16 (attempt 1)
[gemini-3-flash-preview] Prompt 4/16 (attempt 1)
[gemini-3-flash-preview] Prompt 5/16 (attempt 1)


In [22]:
with open("generatedTests.json", "w", encoding="utf-8") as f:
    json.dump(AllTests, f, ensure_ascii=False, indent=2)

In [23]:
def build_judge_prompt(full_context, test_cases):
    return f"""
You are an expert Java test engineer.

TASK:
Select the BEST JUnit 5 test among the candidates.

EVALUATION CRITERIA:
1. Correctness: Matches actual method behavior
2. Coverage: Covers edge cases and branches
3. Assertions: Strong and meaningful assertions
4. Alignment: Must strictly follow the provided class structure and source code
5. Clarity: Clean and maintainable code

STRICT RULES:
- Do NOT assume behavior not present in the code
- Reject trivial or weak tests
- Prefer tests that validate real outcomes
- Only ONE test must be selected

{full_context}

======================
CANDIDATE TESTS
======================

[MODEL 1]
{test_cases[0]}

[MODEL 2]
{test_cases[1]}

[MODEL 3]
{test_cases[2]}

======================
OUTPUT FORMAT (STRICT)
======================
Return ONLY:
1
or
2
or
3
"""

In [24]:
judge_model = "minimax-m2.7:cloud"

num_tests = len(AllTests[0]["test"])

best_tests = []

for i in range(num_tests):
    print(f"\nEvaluating test #{i+1}/{num_tests}")

    full_context = generatedPrompts[i]

    test_cases = [model["test"][i] for model in AllTests]

    prompt = build_judge_prompt(full_context, test_cases)

    try:
        response = chat(
            model=judge_model,
            messages=[{'role': 'user', 'content': prompt}],
        )

        decision = response.message.content.strip().upper()

        if "1" in decision:
            chosen = test_cases[0]
            winner = AllTests[0]["name"]
        elif "2" in decision:
            chosen = test_cases[1]
            winner = AllTests[1]["name"]
        elif "3" in decision:
            chosen = test_cases[2]
            winner = AllTests[2]["name"]
        else:
            chosen = "ERROR: No valid decision"
            winner = "UNKNOWN"

        best_tests.append({
            "index": i,
            "winner_model": winner,
            "test": chosen
        })

    except Exception as e:
        best_tests.append({
            "index": i,
            "winner_model": "ERROR",
            "test": str(e)
        })

print("\nBest test selection completed!")


🧠 Evaluating test #1/16

🧠 Evaluating test #2/16

🧠 Evaluating test #3/16

🧠 Evaluating test #4/16

🧠 Evaluating test #5/16

🧠 Evaluating test #6/16

🧠 Evaluating test #7/16

🧠 Evaluating test #8/16

🧠 Evaluating test #9/16

🧠 Evaluating test #10/16

🧠 Evaluating test #11/16

🧠 Evaluating test #12/16

🧠 Evaluating test #13/16

🧠 Evaluating test #14/16

🧠 Evaluating test #15/16

🧠 Evaluating test #16/16

Best test selection completed!


In [26]:
tests = [item["test"] for item in best_tests]

In [31]:
def ensure_package(test_code, project_filter):
    if not isinstance(test_code, str):
        return test_code

    lines = test_code.strip().splitlines()

    lines = [line for line in lines if not line.strip().startswith("package ")]

    return f"package {project_filter};\n\n" + "\n".join(lines).strip()


tests = []

In [32]:
for item in best_tests:
    test_code = item["test"]

    if not isinstance(test_code, str) or test_code.startswith("ERROR"):
        continue

    test_code = ensure_package(test_code, PROJECT_FILTER)

    tests.append(test_code)

In [33]:
with open("tests.json", "w", encoding="utf-8") as f:
    json.dump(tests, f, ensure_ascii=False, indent=2)

In [34]:
for test in tests :
    print(test)
    print("#"*125)

package com.bank.logic;

import org.junit.jupiter.api.BeforeEach;
import org.junit.jupiter.api.DisplayName;
import org.junit.jupiter.api.Test;
import org.junit.jupiter.params.ParameterizedTest;
import org.junit.jupiter.params.provider.CsvSource;

import java.util.List;

import static org.junit.jupiter.api.Assertions.*;

class SavingsAccountTest {

    private SavingsAccount account;
    private final double INITIAL_BALANCE = 1000.0;
    private final double CUSTOM_RATE = 0.05;

    @BeforeEach
    void setUp() {
        account = new SavingsAccount(INITIAL_BALANCE);
    }

    @Test
    @DisplayName("Constructor with single argument → Sets default interest rate and records history")
    void constructorSingleArg_ShouldSetDefaultRate() {
        // Arrange & Act (done in @BeforeEach)

        // Assert
        assertEquals(INITIAL_BALANCE, account.getBalance());
        assertEquals(0.02, account.getInterestRate(), "Should use DEFAULT_INTEREST_RATE");
        assertEquals(1, account.get